<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<img src="assets/agent03-top-light.svg" alt="With the built-in model MACE-MPA-0, six distorted crystal cells relax in one batch and turn green as each converges, then the same six cells run molecular dynamics together." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 5px solid #76b900; border-radius: 4px; padding: 24px 28px; margin: 8px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="font-variant: small-caps; font-size: 12px; font-weight: 700; letter-spacing: 2px; opacity: 0.75;">AGENTIC TRACK, NOTEBOOK 03</div><h1 style="margin: 6px 0 8px; padding: 0; border: 0;"><span style="font-size: 32px; font-weight: 700; line-height: 1.2; color: inherit;">Ready-made</span></h1><div style="font-size: 18px; line-height: 1.45; opacity: 0.85;">You will learn to relax a batch of crystals and run dynamics on them with a model that is ready the moment you load it.</div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 12px 18px; margin: 12px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; line-height: 1.6;"><b>Hardware:</b> a CUDA-capable NVIDIA GPU with about 10 GB for 200 crystals; fewer crystals fit a smaller one<br><b>Prerequisites:</b> Meet your agent; Many molecules, one batch<br><b>References:</b> <a href="https://nvidia.github.io/nvalchemi-toolkit/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit documentation</a>; <a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit examples</a>; <a href="https://github.com/NVIDIA/skills/tree/main/skills/nvalchemi-toolkit" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">the nvalchemi-toolkit skill</a></div>

<img src="assets/agent03-roadmap-light.svg" alt="The sections of this notebook: relax a batch, simulate, and try it." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

Almost every computed property starts from a relaxed structure: stability,
voltage, adsorption, diffusion. Foundation models such as MACE-MPA-0 work
off the shelf, with no training needed, so a built-in model relaxes
hundreds of structures in minutes.

Toolkit comes with models that are ready to use, and this notebook starts
with them. You will meet these classes along the way:

- `MACEWrapper`: a built-in model, loaded and ready
- `LBFGS`: relaxes every structure in a batch
- `NVE` and `NVTLangevin`: run molecular dynamics on a batch

In [ ]:
# Load the course helpers and print what your agent needs to start
from helpers import agent03 as helpers   # course loaders, plotters, viewers
helpers.configure_tutorial()             # course plot and display defaults
work_dir = helpers.make_work_dir()       # notebooks/agent03_work
n_crystals = 200   # expect about 10 GB of GPU memory; lower it on a smaller GPU
n_temperatures = 12   # copies in the temperature ladder
helpers.show_session()                   # prints the session lines

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">1. Relax a batch with a built-in model</h2>

A structure taken from a database is rarely at the energy minimum of the
model you use. Relaxation moves the atoms downhill until the forces on
them are small. Toolkit relaxes a whole batch at once, and each structure
stops when it has converged.

#### Example: load crystals and a built-in model

Ask your agent to load the course crystals into one batch, load the built-in
MACE-MPA-0 model with cuEquivariance, and report the largest force on each
crystal.

The course crystals are common metals, semiconductors and salts, every atom
displaced by about 0.1 Å.

[cuEquivariance](https://docs.nvidia.com/cuda/cuequivariance/) is NVIDIA's
library of GPU kernels for equivariant networks; MACE uses it for operations
such as its tensor products.

| | |
| --- | --- |
| input  | `helpers.load_crystals(n_crystals)`; MACE-MPA-0, built-in wrapper, cuEquivariance on a GPU |
| output | `crystal_batch`, `mace_model`, `start_max_force`, in `agent03_work/relax_build.py` |

> 💡 **Hint**
>
> `MACEWrapper.from_checkpoint(..., dtype=torch.float32, enable_cueq=device.type == "cuda")`
> loads MACE in the batch's precision and, on a GPU, with the cuEquivariance
> kernels. Neither is the default. On the course GPU, cuEquivariance cut the
> MD run of section 2 on these 200 crystals from 496 to 55 s and the model's
> memory from 33.5 to 8.2 GiB.

In [ ]:
%load agent03_work/relax_build.py

In [ ]:
# Show the crystals and the largest force on each
helpers.show_crystal_grid(crystal_batch, start_max_force)

Each box is one crystal, coloured by the largest force on any of its atoms.

> </> API
>
> `MACEWrapper.from_checkpoint(name_or_path, device=...)`
>
> **Input:** a checkpoint name or path, and a device<br>
> **Effect:** loads a pretrained MACE model with frozen weights, in eval mode

#### Example: relax the batch

Ask your agent to relax every crystal in the batch and to time the batched
run beside one-by-one relaxation of three of them.

| | |
| --- | --- |
| input  | `crystal_batch`; `mace_model` |
| output | `relaxed_batch`, `relaxed_max_force`, `force_threshold`, `batch_seconds`, `single_seconds`, in `agent03_work/relax_run.py` |

In [ ]:
%load agent03_work/relax_run.py

In [ ]:
# Compare the largest forces before and after the relaxation
helpers.plot_force_histograms(start_max_force, relaxed_max_force, force_threshold)

In [ ]:
# Compare the batched run with relaxing one by one
helpers.plot_batch_speedup(batch_seconds, single_seconds, len(start_max_force))

The one-by-one bar is projected from three structures.

> ⚙️ **Fine control**
>
> Your agent proposes the settings. To choose one yourself, look at the
> parameters and name the exact value in your prompt.
>
> ```python
> LBFGS(
>     model=model,
>     history_size=...,   # past steps the optimizer remembers
>     maxstep=...,        # largest move of an atom per step
>     n_steps=...,        # step budget
>     convergence_hook=ConvergenceHook.from_fmax(0.05),
> )                       # stop below this force, eV/Å
> ```
>
> Example: `relax again, fmax 0.01 eV/Å, at most 1000 steps`

> </> API
>
> `FusedStage(sub_stages=[(0, opt)])`
>
> **Input:** the optimizer as stage 0<br>
> **Effect:** gives every system a `status` and retires each one as it converges

> </> API
>
> `model.make_neighbor_hooks()`
>
> **Input:** none; called on the model<br>
> **Effect:** hooks that rebuild the neighbor list before each force call; register them before `run()`

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">2. Simulate with a built-in model</h2>

Molecular dynamics (MD) moves the atoms forward in time under the model's
forces. Without a thermostat the total energy should stay constant. A model
and timestep that cannot keep it constant in a short run cannot be trusted
in a long one, and many processes of interest take long.

#### Example: prepare the MD run

Ask your agent to give the relaxed crystals starting velocities for the MD
run.

| | |
| --- | --- |
| input  | `relaxed_batch`; `mace_model` |
| output | `md_batch`, `start_temperature`, in `agent03_work/md_build.py` |

> 💡 **Hint**
>
> A batch remembers the run it came from. Before the MD starts, ask your
> agent what state `relaxed_batch` is in, and whether the atoms will move.

In [ ]:
%load agent03_work/md_build.py

In [ ]:
# Show the starting temperature of each crystal
helpers.plot_start_temperature(start_temperature)

Each crystal starts near the temperature your agent proposed.

> </> API
>
> `initialize_velocities(v, masses, T, batch_idx_int32, ...)`
>
> **Input:** velocities, masses, a temperature in K and the atom-to-structure index as int32<br>
> **Effect:** thermal starting velocities

#### Example: a short MD with no thermostat

Ask your agent to run the batch with no thermostat and to save the total
energy of each crystal over time.

| | |
| --- | --- |
| input  | `md_batch`; `mace_model` |
| output | `time_fs`, `crystal_energy`, in `agent03_work/md_run.py` |

In [ ]:
%load agent03_work/md_run.py

In [ ]:
# Show the change in total energy of each crystal
helpers.plot_energy_drift(time_fs, crystal_energy, md_batch)

Flat lines mean the energy is conserved for that crystal.

> ⚙️ **Fine control**
>
> ```python
> NVE(model=model, dt=..., n_steps=...)             # dt in fs
> NVTLangevin(
>     model=model,
>     dt=...,            # fs
>     temperature=...,   # K, one value or one per system
>     friction=...,      # 1/fs
>     n_steps=...,
> )
> ```
>
> Example: `run again with half the timestep`

#### Try it: a ladder of temperatures in one batch

A thermostat holds each structure at its own temperature. Ask your agent
to run copies of one crystal in a single batch, each at a different
temperature, from cold to far too hot, and to record the potential energy
per atom of each along the run.

| | |
| --- | --- |
| input  | a relaxed crystal of your own, or `helpers.load_ladder_crystal()`; `mace_model`; `n_temperatures` |
| output | `time_fs`, `replica_temperature`, `replica_energy`, in `agent03_work/temperature_ladder.py` |

Done when: the plot shows one energy line for each temperature, hotter
copies above cooler ones by the end of the run.

> 💡 **Hint**
>
> A batch can vary settings as well as structures. In this run:
> `positions`, `atomic_numbers`: the same crystal in every copy;
> `temperature`: one value per copy, shape `(n_temperatures,)`, in K.

> 💡 **Hint**
>
> No crystal at hand? `helpers.load_ladder_crystal()` returns a 64-atom
> copper crystal that needs no relaxation.

In [ ]:
%load agent03_work/temperature_ladder.py

In [ ]:
# Plot each copy's energy along the run, cold to hot
helpers.plot_temperature_ladder(time_fs, replica_temperature, replica_energy)

#### Try it: turn the knobs for best performance

You have already used one of the fastest MACE setups, and it is not the
default one: cuEquivariance on, model compilation off. Here you switch both
settings to see what each does to the run time. Ask your agent to time ten
L-BFGS steps on 50 crystals of your batch, the same 50 in every case because
without cuEquivariance the full batch needs about four times the memory,
with cuEquivariance alone, compilation alone, both and neither, and to
explain what each setting is for.

| | |
| --- | --- |
| input  | `crystal_batch`; MACE-MPA-0 |
| output | `speed_table`, `setting_roles`, in `agent03_work/speed_settings.py` |

Done when: the table shows a first-call time and a repeat time for each of the four cases.

> 💡 **Hint**
>
> `helpers.show_speed_table(...)` draws the result. `speed_table`: a pandas
> DataFrame, one row per case, with the columns `cueq` and `compile` (True
> or False), `first_call_s` and `repeat_s` (s); `setting_roles`: a dict
> with a text for `cueq` and one for `compile`.

In [ ]:
%load agent03_work/speed_settings.py

In [ ]:
# Show the timings and the role of each setting
helpers.show_speed_table(speed_table, setting_roles)

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">Explore more</h2>

Official Toolkit examples that take relaxation and dynamics further:

<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 4px 0 18px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/05_custom_integrator.py">Write your own integrator</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Subclass <code>BaseDynamics</code> and override <code>pre_update</code> and <code>post_update</code>.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/intermediate/03_npt_barostat_validation.py">Hold the pressure</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;"><code>NPT</code> couples the cell to the model's stress at a set pressure.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/09_uma_nve.py">Another ready-made model</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;"><code>UMAWrapper</code> loads UMA, one checkpoint with heads for molecules and crystals.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/basic/04_nve_energy_conservation.py">Watch a run</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;"><code>EnergyDriftMonitorHook</code> warns on drift; <code>LoggingHook</code> records energies as it runs.</div></div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; padding: 15px 20px; margin: 24px 0 18px; line-height: 1.6;"><h2 style="margin: 0 0 10px; padding: 0; border: 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 24px; font-weight: 500; line-height: 1.25; color: inherit;">Notebook 03 recap</h2><div style="margin: 0 0 10px;">A built-in model needs no preparation: you loaded it, relaxed a batch of crystals and ran dynamics on them. The next notebook is for when a ready-made model needs more.</div><div style="margin: 0;"><img src="assets/fork-agent-light.svg" alt="" width="18" height="18" style="width: 18px; height: 18px; vertical-align: -3px; margin-right: 6px;"><b>Next:</b> <a href="alchemi-agent-04-made-to-order.ipynb" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Made to order →</a></div></div>